# Lesson 2: Prompt Evaluation

### Prompt Engineering
- a set of best practices and guidance to improve your prompts
    - multishot prompting
    - structuring with XML tags

### Prompt Evaluation
- automated testing to measure how well your prompts work
    - test against expected answers
    - compare different versions of the same prompt
    - review outputs for errors

#### Best option for drafting a prompt:
- run your prompt through an evaluation pipeline to score it, then iterate on the prompt

## 2.1 Prompt Eval Workflow
### Five Key Steps:
1) Draft a prompt
2) Create an Eval dataset 
    - contains qns that we will merge with our prompt
    - can be done by hand or generated by Claude
3) Feed through Claude 
4) Feed through a Grader
5) Change prompt and repeat

- key benefit of this workflow is getting objective measurements of prompt performance
    - allows us to compare different porompt versions numerically
    - enables us to determine the version with the best score
    - continue iterating to find better approaches 

## 2.2 Generating Test Datasets
### Goal:
- Write a prompt that will assist users in writing Python code, JSON config, or Regular Expressions focused on AWS-specific use cases
- Input:
    - user will request code for a specific task
- Output:
    - Python, JSON or a regular expressions without any explanation 

### Eval Dataset:
- each object contains a "task" that we will merge into the prompt



In [12]:
# Load env variables
from dotenv import load_dotenv
load_dotenv()

True

In [13]:
# Create API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [37]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)
    
def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model":model,
        "max_tokens":500,
        "messages":messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }
    
    if system:
        params["system"] = system
        
    message = client.messages.create(**params)
    return message.content[0].text


In [47]:
import json

def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. 
Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.
    
Example output:
```json
[
  {
    "task": "Description of task",
    "format": "json" or "python" or "regex",
    "solution_criteria": "Key criteria for evaluating the solution"
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.

"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)
    

In [48]:
dataset = generate_dataset()
dataset

[{'task': "Extract AWS service names from CloudFormation template resource types (e.g., 'AWS::S3::Bucket', 'AWS::Lambda::Function')",
  'format': 'regex',
  'solution_criteria': 'Regex pattern should correctly extract the service name (second component) from AWS CloudFormation resource type strings, handling various AWS services'},
 {'task': 'Parse an AWS S3 bucket access log line and extract the bucket owner, bucket name, request time, source IP, and HTTP status code',
  'format': 'python',
  'solution_criteria': 'Function should parse a space-delimited S3 access log line and return a dictionary with keys: bucket_owner, bucket_name, request_time, source_ip, http_status_code'},
 {'task': 'Create a JSON policy document for an AWS IAM role that allows EC2 instances to read objects from a specific S3 bucket',
  'format': 'json',
  'solution_criteria': 'Valid JSON policy document with correct structure, AssumeRolePolicyDocument format, allowing s3:GetObject action on specified bucket resou

Save dataset and savei it to a file so we can easily load it later during eval

In [49]:
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

## 2.3 Running the Eval

In [43]:
# Function takes a test case and merges it with our prompt template
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain REGEX
* Do not add any comments or commentary or explanation
"""
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [50]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10


Respond with JSON. Keep your response concise and direct. Any quotes in the JSON must be escaped.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    
    print("MODEL RESPONSE:")
    print(eval_text)
    
    return json.loads(eval_text)

## 2.4 Code Based Grading

In [44]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [45]:
# function orchestrates running a single test case and grading the result
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    model_grade = grade_by_model(test_case, output)
    mode_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    syntax_score = grade_syntax(output, test_case)
    
    score = (mode_score + syntax_score) / 2
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [51]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

Running the evaluation

In [52]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

MODEL RESPONSE:

{
    "strengths": [
        "Correctly extracts the service name component from standard AWS CloudFormation resource types using a well-formed regex pattern",
        "Handles both JSON string and dictionary inputs gracefully with type checking",
        "Returns sorted results for consistent, predictable output"
    ],
    "weaknesses": [
        "Regex pattern `[a-zA-Z0-9]+` is too restrictive and fails to match AWS services with hyphens (e.g., AWS::ApiGateway::RestApi, AWS::ElastiCache::CacheCluster, AWS::Events::Rule)",
        "Does not validate that extracted services are legitimate AWS services, allowing invalid matches",
        "Lacks error handling for malformed JSON or missing 'Resources' key, which could cause silent failures"
    ],
    "reasoning": "The solution correctly parses CloudFormation templates and implements the core extraction logic, but the regex pattern has a critical flaw. AWS service names commonly contain hyphens (ApiGateway, ElastiCache,

Examining the results:

In [ ]:
print(json.dumps(results, indent=2))

## 2.5 Code based grading
- add functions to validate JSON/Python/Regex
- ensure dataset test cases indicate the type of generated content 
- update draft prompt to make it clear we only want the relevant JSON/Python/Regex
- merge scores from the model grader and the code grader